### Inspect PC variation between sparse lms, dense corresp, NSM latents
---
Do spearman's rank heatmap and PC traversal grid plot. Need to run save_pc_snapshots.py and pc_snapshot_grid.py to build NSM traversal plot, then stitch it together with lm based warp plots below.

### 1. Setup and paths

In [ ]:
# Imports and paths
import os, re, json
import subprocess, sys
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pyvista as pv
from NSM.morphometrics import procrustes_dist, gm_prcomp
from NSM.plotting import (load_mrk_json, make_renderers, view_rotation, build_warp_grid,
                          stitch_grid, load_rgb, load_pc_block, pc_block, spearman_matrix,
                          plot_spearman_heatmaps, spearman_tables)

# Specify training directory and atlas directory
RUN          = "run_v72"                      # training attempt directory
ATLAS_RUN    = "2026_07-15_13_06_22/"  # atlas/builder run that produced alignedLMs
DROPBOX_ROOT = Path("/home/k.wolcott/UFL Dropbox/Katherine Wolcott/neural_shape_models/final_dataset_aug26/atlas/")

# Build other directories relative to those above
cwd      = Path.cwd()
base_wd  = cwd.parent
train_dir = base_wd / RUN
os.chdir(train_dir)
print(f"Working directory: {os.getcwd()}")

LM_DIR       = DROPBOX_ROOT / ATLAS_RUN / "alignedLMs"
ATLAS_DIR    = DROPBOX_ROOT / ATLAS_RUN / "atlas"
MEAN_LMS_FN  = ATLAS_DIR / "atlas_sparse_landmarks.mrk.json"
MEAN_MESH_FN = ATLAS_DIR / "atlas_model.ply"
atlas_mesh = pv.read(str(MEAN_MESH_FN))

OUT_DIR = Path("spearman_mesh_grid")
OUT_DIR.mkdir(exist_ok=True)
print(f"Outputs will be written to: {OUT_DIR.resolve()}")

# Latent traversal figure
AMPLIFY = 1.5      # PC score range multiplier; 1.0 = observed range only

# Load config and parse species / vertebra from filenames (same logic as PCA_tSNE_UMAP.ipynb)
config_path = "model_params_config.json"
with open(config_path) as f:
    cfg = json.load(f)
print(f"\033[92mLoaded config from {config_path}\033[0m")

# Parse filenames
train_paths   = cfg["list_mesh_paths"]
all_vtk_files = [os.path.basename(f) for f in train_paths]
print(f"{len(all_vtk_files)} meshes listed in config")

pat = re.compile(r"^(?P<species>[\w\s\-]+)[\-_ ]+[\w\d]+[\-_ ]+(?P<vertebra>[CTL]?\d+)", re.IGNORECASE)
labels, unmatched_files = [], []
for f in all_vtk_files:
    m = pat.match(os.path.basename(f))
    if m:
        labels.append((m.group("species").strip(), m.group("vertebra").strip()))
    else:
        labels.append((None, None))
        unmatched_files.append(f)

print(f"Parsed {sum(1 for s, v in labels if s)} / {len(labels)} filenames")
if unmatched_files:
    print(f"\033[33mUnmatched ({len(unmatched_files)}), first 5:\033[0m", unmatched_files[:5])

## Load landmarks, check data, run PCA (like R geomorph prcomp)

### Load landmarks (`.mrk.json`) and build the shape array

In [ ]:
# Load 3D Slicer Atlas aligned and scaled landmark data
lm_coords = []
for fpath in all_vtk_files:
    lm_name = os.path.splitext(fpath)[0] + ".mrk.json"
    lm_path = LM_DIR / lm_name
    coords, _ = load_mrk_json(lm_path)
    lm_coords.append(coords)

lm_coords_3d = np.stack(lm_coords)   # (N, p, 3)
print(f"Landmark data shape - 3d: {lm_coords_3d.shape}")

# Atlas mean sparse landmarks (same landmark set as LM_DIR, from the setup cell)
mean_lms_3d, _ = load_mrk_json(MEAN_LMS_FN)
print(f"Atlas mean landmarks shape: {mean_lms_3d.shape}")
assert mean_lms_3d.shape == lm_coords_3d.shape[1:], (
    f"Atlas has {mean_lms_3d.shape[0]} landmarks but specimens have {lm_coords_3d.shape[1]} "
    f"-- wrong atlas run, or dense vs sparse mismatch")

In [ ]:
# Principal components of the Procrustes coordinates
pca = gm_prcomp(lm_coords_3d)
print(f"{pca['x'].shape[1]} non-trivial PCs from {pca['p']*pca['k']} coordinates")
for i in range(min(10, len(pca["prop"]))):
    print(f"  PC{i+1}: {100*pca['prop'][i]:6.2f}%   cumulative {100*pca['cum'][i]:6.2f}%")

## Spearman rank correlations

In [ ]:
# ── Load and merge PC score tables from  PCA_tSNE_UMAP_paper_figs.ipynb───────────────────────────────────────────
ID_COLS  = ["specimen_id", "vertebra"]
STATS_DIR = train_dir / "pca_tsne_umap_results"

sparse_df = load_pc_block(STATS_DIR / "pc_sparse_points_for_stats.csv", "PC_sparse", ID_COLS)
dense_df  = load_pc_block(STATS_DIR / "pc_dense_points_for_stats.csv",  "PC_dense",  ID_COLS)
latent_df = load_pc_block(STATS_DIR / "pc_latent_points_for_stats.csv", "PC_latent", ID_COLS)

merged = sparse_df.merge(dense_df, on=ID_COLS).merge(latent_df, on=ID_COLS)
n = len(merged)
print(f"{n} specimens matched across sparse / dense / latent representations")
if n < len(sparse_df):
    print(f"  ↳ {len(sparse_df) - n} specimens dropped (absent from ≥1 representation)")

sparse_cols, X_sparse = pc_block(merged, "PC_sparse")
dense_cols,  X_dense  = pc_block(merged, "PC_dense")
latent_cols, X_latent = pc_block(merged, "PC_latent")

PAIRS = [("LANDMARKS",     X_sparse, "DENSE CORRESP", X_dense),
         ("LANDMARKS",     X_sparse, "NSM LATENTS",   X_latent),
         ("DENSE CORRESP", X_dense,  "NSM LATENTS",   X_latent)]

RESULTS = [(a, b, *spearman_matrix(A, B)) for a, A, b, B in PAIRS]

In [ ]:
# ── Figure and tables ────────────────────────────────────────────────────────
plot_spearman_heatmaps(RESULTS, out_path=OUT_DIR / "spearman_pc_correlations.png")

plt.show()

for lbl_a, lbl_b, R, P in RESULTS:
    spearman_tables(lbl_a, lbl_b, R, P, out_dir=OUT_DIR)

print(f"\nAll outputs → {OUT_DIR.resolve()}")

## PC Warp Grids - Landmarks and NSM

### Use LMs to warp mean mesh and save snapshots along PC's

In [ ]:
# Save snapshots
N_PCS   = 4     # rows  (PC1 … PC4)
N_STEPS = 4     # columns per row
WIDTH   = 640   # px per panel, matches snapshot script
HEIGHT  = 480
BASE_BG_COL     = np.array([0.839, 1, 0.996])    # aquamarine
MAX_TINT_BG_COL = np.array([0, 0.58, 0.522])  # dark teal
BG_COLORS = np.linspace(BASE_BG_COL, MAX_TINT_BG_COL, N_PCS)  # one row per PC
VIEW_ROT_DEG = {"side": 13.0, "front": 90.0 + 13.0}
RENDERERS = make_renderers(WIDTH, HEIGHT)

for view in ("front", "side"):
    build_warp_grid(pca, mean_lms_3d, atlas_mesh,
                    out_dir=f"pc_snapshots/sparseLMs/{view}",
                    label=f"sparseLMs_{view}",
                    renderers=RENDERERS,
                    rot_matrix=view_rotation(VIEW_ROT_DEG[view]),
                    n_pcs=N_PCS, n_steps=N_STEPS,
                    width=WIDTH, height=HEIGHT,
                    bg_cols=BG_COLORS, 
                    amplify=AMPLIFY)

In [ ]:
# Stitch snapshots together 
FLIP_IDX = []

for view in ("front", "side"):
    grid_path=OUT_DIR / f"pc_grid_sparseLMs_{view}.png"
    stitch_grid(Path(f"pc_snapshots/sparseLMs/{view}"),
                label=f"sparseLMs_{view}",
                grid_path=grid_path,
                flip_pcs=FLIP_IDX)
    plt.figure(figsize=(10, 7.5))
    plt.imshow(load_rgb(grid_path))
    plt.axis("off")
    plt.title(f"sparseLMs — {view}")
    plt.show()

### Use NSM to reconstruct meshes along PC's

In [ ]:
# Save snapshots along PCs for front and side views
N_PCS = 4

for view in ("front", "side"):
    for pc in range(1, N_PCS + 1):
        proc = subprocess.Popen([sys.executable, "-W", "ignore",
                                 str(base_wd / "data_viz" / "save_pc_snapshots.py"),
                                 "--pc", str(pc), "--view", view,
                                 "--train-dir", str(train_dir), 
                                 "--amplify", str(AMPLIFY)],
                                stdout=subprocess.PIPE, text=True, bufsize=1)
        for line in proc.stdout:
            if line.lstrip().startswith(("NSM", "Done", "Error")):
                print(line, end="", flush=True)
        proc.wait()

In [ ]:
# Stitch snapshots together
FLIP_PCS = [1,3,4]      # 1-based; PC sign is arbitrary, flipped to match the landmark rows

for view in ("front", "side"):
    grid_path = OUT_DIR / f"pc_grid_nsm_{view}.png"
    subprocess.run([sys.executable, str(base_wd / "data_viz" / "stitch_pc_snapshots.py"),
                    "--view", view,
                    "--train-dir", str(train_dir),
                    "--flip_pc_ax", *map(str, FLIP_PCS),
                    "-o", str(grid_path)], check=True)
    plt.figure(figsize=(10, 7.5))
    plt.imshow(load_rgb(grid_path))
    plt.axis("off")
    plt.title(f"NSM — {view}")
    plt.show()

#### Render NSM and LM plots side by side to evaluate if any PC axes need to be flipped for easier comparison

In [ ]:
# Render NSM and lm plots

for view in ("front", "side"):
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    for ax, label in zip(axes, ("sparseLMs", "nsm")):
        ax.imshow(load_rgb(OUT_DIR / f"pc_grid_{label}_{view}.png"))
        ax.set_title(f"{label} — {view}")
        ax.axis("off")
    plt.show()

## Stitch LM and NSM based grids into figure panel for paper

In [ ]:
# Stitch traversal plots together
FONT_SIZE        = 15
LABEL_FONT_SIZE  = FONT_SIZE
PC_FONT_SIZE     = FONT_SIZE
DPI              = 300
LW               = 0.5

SPARSE_SIDE  = OUT_DIR / "pc_grid_sparseLMs_side.png"
SPARSE_FRONT = OUT_DIR / "pc_grid_sparseLMs_front.png"
NSM_SIDE     = OUT_DIR / "pc_grid_nsm_side.png"
NSM_FRONT    = OUT_DIR / "pc_grid_nsm_front.png"
OUT_FIGURE   = OUT_DIR / "fig_m_pc_traversal.png"

N_PCS      = 4
COL_LABELS = ["BACK", "SIDE"]
ROW_LABELS = ["LANDMARKS", "NSM LATENTS"]

GRID = [[SPARSE_FRONT, SPARSE_SIDE ],
        [NSM_FRONT,    NSM_SIDE    ]]


# ── Height ratios ─────────────────────────────────────────────────────────────
grid_ratios = [load_rgb(row[0]).shape[0] / load_rgb(row[0]).shape[1] for row in GRID]

# ── Figure ────────────────────────────────────────────────────────────────────
plt.rcParams.update({"font.size":      FONT_SIZE,
                     "font.weight":    "normal",
                     "axes.linewidth": LW,
                     "text.color":     "black",})

fig = plt.figure(figsize=(12, sum(grid_ratios) * 4), dpi=DPI)

gs = gridspec.GridSpec(2, 2,
                       figure=fig,
                        height_ratios=grid_ratios,
                        hspace=0.06,
                        wspace=0.03,
                        left=0.12, right=0.995,
                        top=0.97,  bottom=0.005)

for r, row_paths in enumerate(GRID):
    for c, fpath in enumerate(row_paths):
        img = load_rgb(fpath)
        ax  = fig.add_subplot(gs[r, c])
        ax.imshow(img, aspect="auto")
        ax.set_xticks([])
        ax.set_yticks([])

        for sp in ax.spines.values():
            sp.set_visible(True)
            sp.set_linewidth(LW)
            sp.set_edgecolor("black")

        # Column headers on first row only
        if r == 0:
            ax.set_title(COL_LABELS[c], fontsize=LABEL_FONT_SIZE,
                         fontweight="normal", pad=10)

        # Row label on left column
        if c == 0:
            ax.set_ylabel(ROW_LABELS[r], fontsize=LABEL_FONT_SIZE,
                          fontweight="normal", labelpad=40)
            # PC labels
            for pc_idx in range(N_PCS):
                y = 1 - (pc_idx + 0.5) / N_PCS
                ax.text(-0.01, y, f"PC{pc_idx + 1}",
                        transform=ax.transAxes,
                        fontsize=PC_FONT_SIZE, fontweight="normal",
                        va="center", ha="right", color="black")

plt.savefig(str(OUT_FIGURE), dpi=DPI, bbox_inches="tight")
plt.close()
print(f"Saved → {OUT_FIGURE}")

plt.figure(figsize=(10, 10 * sum(grid_ratios) / 2))
plt.imshow(load_rgb(OUT_FIGURE))
plt.axis("off")
plt.show()